# CC18 figures

Reads the results directories written by `experiments.ipynb` and renders the paper figures
into `<results_dir>/figures/` (PNG + PDF). Run the experiments notebook first.

Set `RESULTS_DIR` below to the run you want to plot (max-aggregation is the main protocol).

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import openml_flow as wf  # makes ShapFoldResult importable when unpickling artifacts

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["savefig.bbox"] = "tight"

RESULTS_DIR = Path("results_cc18_max")
SPLIT = "task_group_kfold"
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

REPRESENTATION_LABELS = {"tfidf_meta": "TF-IDF + meta", "minilm_meta": "MiniLM + meta", "meta_only": "Meta only"}

def experiment_label(x):
    return REPRESENTATION_LABELS.get(str(x), str(x))

def run_dirs(root: Path):
    """Yield (experiment_name, run_dir) for every run directory under ``root``."""
    for meta in sorted(root.rglob("run_metadata.json")):
        import json
        name = json.loads(meta.read_text()).get("name", meta.parent.name)
        yield name, meta.parent

def savefig(fig, stem):
    fig.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
    print("saved", FIG_DIR / f"{stem}.png")

list(run_dirs(RESULTS_DIR))

## Load consolidated tables

In [ ]:
anchor_df = pd.read_csv(RESULTS_DIR / "predictive_anchor.csv")
anchor_df["Representation"] = anchor_df["experiment"].map(experiment_label)

local_path = RESULTS_DIR / "local_shap_metrics_summary.csv"
local_df = pd.read_csv(local_path) if local_path.exists() else pd.DataFrame()
if not local_df.empty:
    local_df["Representation"] = local_df["experiment"].map(experiment_label)

anchor_df

## Figure 1 — Predictive anchor (R² by representation and model)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=anchor_df, x="Representation", y="r2_mean", hue="model", ax=ax)
ax.set_ylabel("R² (task-group CV)")
ax.set_xlabel("")
ax.set_title("Predictive performance by representation")
savefig(fig, "figure1_predictive_anchor")

## Figure 2 — Top global SHAP features

In [ ]:
TOP_K = 12
frames = []
for name, rd in run_dirs(RESULTS_DIR):
    for p in rd.glob(f"global_shap_{SPLIT}_*.csv"):
        model = p.stem[len(f"global_shap_{SPLIT}_"):]
        df = pd.read_csv(p).head(TOP_K)
        df["experiment"] = name
        df["model"] = model
        frames.append(df)

if frames:
    global_df = pd.concat(frames, ignore_index=True)
    global_df["feature_short"] = (
        global_df["feature"].str.replace("text::", "", regex=False).str.replace("meta::", "", regex=False)
    )
    g = sns.catplot(
        data=global_df, kind="bar", y="feature_short", x="mean_abs_shap",
        row="model", col="experiment", sharey=False, height=4, aspect=1.1,
    )
    g.set_titles("{col_name} | {row_name}")
    g.fig.suptitle("Top global SHAP features", y=1.02)
    savefig(g.fig, "figure2_global_shap")
else:
    print("No global SHAP CSVs found — run experiments with compute_shap=True.")

## Figure 3 — Local explanation quality (readability and sparsity)

In [ ]:
if not local_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    sns.barplot(data=local_df, x="Representation", y="readability_mean", hue="model", ax=axes[0])
    axes[0].set_title("Readability")
    sns.barplot(data=local_df, x="Representation", y="sparsity_mean", hue="model", ax=axes[1])
    axes[1].set_title("Sparsity")
    for ax in axes:
        ax.set_xlabel("")
    savefig(fig, "figure3_local_quality")
else:
    print("No local SHAP metrics summary found.")

## Figure 4 — Global top-k stability across folds

In [ ]:
rows = []
for name, rd in run_dirs(RESULTS_DIR):
    for p in rd.glob(f"global_shap_stability_{SPLIT}_*.csv"):
        model = p.stem[len(f"global_shap_stability_{SPLIT}_"):]
        df = pd.read_csv(p)
        rows.append({"experiment": name, "model": model, "jaccard_mean": df["jaccard"].mean()})

if rows:
    stab_df = pd.DataFrame(rows)
    stab_df["Representation"] = stab_df["experiment"].map(experiment_label)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=stab_df, x="Representation", y="jaccard_mean", hue="model", ax=ax)
    ax.set_title("Fold-to-fold stability of global top-k features")
    ax.set_ylabel("mean Jaccard")
    ax.set_xlabel("")
    savefig(fig, "figure4_global_stability")
else:
    print("No stability CSVs found.")

## Figure 5 — Interaction ratio (optional; requires compute_shap_interactions=True)

In [ ]:
rows = []
for name, rd in run_dirs(RESULTS_DIR):
    for p in rd.glob(f"interaction_metrics_{SPLIT}_*.csv"):
        model = p.stem[len(f"interaction_metrics_{SPLIT}_"):]
        df = pd.read_csv(p)
        rows.append({"experiment": name, "model": model, "interaction_ratio": df["interaction_ratio"].mean()})

if rows:
    inter_df = pd.DataFrame(rows)
    inter_df["Representation"] = inter_df["experiment"].map(experiment_label)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=inter_df, x="Representation", y="interaction_ratio", hue="model", ax=ax)
    ax.set_title("Mean SHAP interaction ratio")
    ax.set_xlabel("")
    savefig(fig, "figure5_interaction_ratio")
else:
    print("No interaction metrics found (set compute_shap_interactions=True to produce them).")

## Figure 6 — Case-study local explanation

Top-k signed SHAP contributions for one held-out instance, loaded from `shap_artifacts.pkl`.

In [ ]:
CASE_EXPERIMENT = "tfidf_meta"
CASE_MODEL = "hist_gbrt"
CASE_INSTANCE = 0
TOP_K = 10

pkl = RESULTS_DIR / CASE_EXPERIMENT / "shap_artifacts.pkl"
if pkl.exists():
    with open(pkl, "rb") as f:
        artifacts = pickle.load(f)
    fold0 = artifacts[SPLIT][CASE_MODEL]["shap_artifacts"][0]
    row = fold0.shap_values[CASE_INSTANCE]
    idx = np.argsort(np.abs(row))[::-1][:TOP_K]
    plot_df = pd.DataFrame({
        "feature": [fold0.feature_names[i].replace("text::", "").replace("meta::", "") for i in idx],
        "shap_value": [float(row[i]) for i in idx],
    })
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=plot_df, x="shap_value", y="feature", orient="h", ax=ax)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"Local SHAP: {experiment_label(CASE_EXPERIMENT)} / {CASE_MODEL} (row {fold0.row_ids[CASE_INSTANCE]})")
    savefig(fig, "figure6_local_case_study")
else:
    print(f"No shap_artifacts.pkl in {pkl.parent} — run experiments with compute_shap=True.")